# POC — Extração Estruturada de Documentos
**Desafio Técnico | Tech for Humans | Estágio em P&D — IA Generativa**  
**Candidata:** Ana Beatriz Araújo Silva  
**Data:** 01 de julho de 2026  

---

Esta Prova de Conceito demonstra a abordagem recomendada no Dossiê de Pesquisa: substituição do pipeline de duas inferências por uma **única chamada ao modelo** com **structured outputs nativos** (`response_schema` nativo da API Gemini).

**Modelo utilizado na POC:** `gemini-2.5-flash` (Google DeepMind)

> **Nota sobre acesso à API:** O Gemini 2.5 Flash está disponível no free tier do Google AI Studio com limite de 1.500 requisições/dia e 1M tokens/min — suficiente para validação desta POC. A chave de API é obtida gratuitamente em [aistudio.google.com](https://aistudio.google.com).

## 0. Instalação e Imports

In [3]:
# !pip install google-genai pypdf

from google import genai
from google.genai import types
import json, time, base64, os
from pathlib import Path

print("Tudo importado com sucesso!")
print(f"SDK google-genai versão: {genai.__version__}")

Tudo importado com sucesso!
SDK google-genai versão: 2.10.0


## 1. Configuração da API

In [4]:
from dotenv import load_dotenv

load_dotenv()  # carrega o .env automaticamente

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL  = "gemini-2.5-flash"

print(f"Cliente configurado. Modelo: {MODEL}")

Cliente configurado. Modelo: gemini-2.5-flash


## 2. Teste de Conexão

In [5]:
response = client.models.generate_content(
    model=MODEL,
    contents="Responda apenas: conexão ok"
)
print(response.text)

conexión ok


## 3. Funções Auxiliares

In [6]:
import pypdf, io

def carregar_imagem(caminho: str) -> tuple:
    """Carrega imagem como bytes + MIME type."""
    ext = Path(caminho).suffix.lower()
    mime_map = {'.jpg': 'image/jpeg', '.jpeg': 'image/jpeg',
                '.png': 'image/png',  '.webp': 'image/webp'}
    mime_type = mime_map.get(ext, 'image/jpeg')
    with open(caminho, 'rb') as f:
        dados = f.read()
    return dados, mime_type


def carregar_pdf_parcial(caminho: str, max_paginas: int = 10) -> bytes:
    """Retorna as primeiras N páginas do PDF como bytes."""
    reader = pypdf.PdfReader(caminho)
    writer = pypdf.PdfWriter()
    total  = min(max_paginas, len(reader.pages))
    for i in range(total):
        writer.add_page(reader.pages[i])
    buf = io.BytesIO()
    writer.write(buf)
    print(f"Páginas enviadas: {total} de {len(reader.pages)}")
    return buf.getvalue()

print("Funções auxiliares carregadas.")

Funções auxiliares carregadas.


In [7]:
# Preços públicos Gemini 2.5 Flash — junho 2026
# Input (texto + imagem): $0.15 / 1M tokens
# Output:                 $0.60 / 1M tokens
# Fonte: https://ai.google.dev/gemini-api/docs/pricing
PRECO_INPUT_POR_M  = 0.15
PRECO_OUTPUT_POR_M = 0.60

MAX_TENTATIVAS = 3


def extrair_com_schema(conteudo: list, schema: dict, caso: str,
                       thinking_budget: int = 0,
                       max_output_tokens: int = 8192) -> tuple:
    """
    Extração estruturada em UMA única inferência usando response_schema nativo do Gemini.
    Inclui retry automático com backoff exponencial para erros 503.

    Args:
        conteudo:          lista de parts (texto, imagem, PDF)
        schema:            JSON Schema dos campos a extrair
        caso:              rótulo para log
        thinking_budget:   0 = desativado; >0 = raciocínio incremental
        max_output_tokens: limite de tokens de saída

    Returns:
        (resultado_dict, metricas_dict)
    """
    print(f"\n{'='*60}")
    print(f"  {caso}")
    print(f"{'='*60}")

    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=schema,
        thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget),
        max_output_tokens=max_output_tokens,
    )

    inicio = None
    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            print(f"Tentativa {tentativa}/{MAX_TENTATIVAS} "
                  f"(thinking_budget={thinking_budget})...")
            inicio = time.time()
            response = client.models.generate_content(
                model=MODEL,
                contents=conteudo,
                config=config,
            )
            break  # sucesso
        except Exception as e:
            print(f"  Erro: {e}")
            if tentativa == MAX_TENTATIVAS:
                raise
            espera = 15 * tentativa
            print(f"  Aguardando {espera}s antes de tentar novamente...")
            time.sleep(espera)

    latencia = round(time.time() - inicio, 2)
    resultado = json.loads(response.text)

    tokens_input  = response.usage_metadata.prompt_token_count
    tokens_output = response.usage_metadata.candidates_token_count
    custo_usd = (
        tokens_input  * PRECO_INPUT_POR_M +
        tokens_output * PRECO_OUTPUT_POR_M
    ) / 1_000_000

    print(f"  Latência:      {latencia}s")
    print(f"  Tokens input:  {tokens_input}")
    print(f"  Tokens output: {tokens_output}")
    print(f"  Custo est.:    ${custo_usd:.6f} USD")
    print(f"  Inferências:   1 (pipeline unificado)")

    return resultado, {
        "latencia_s":      latencia,
        "tokens_input":    tokens_input,
        "tokens_output":   tokens_output,
        "custo_usd":       round(custo_usd, 6),
        "inferencias":     1,
        "thinking_budget": thinking_budget,
    }

print("Função de extração principal carregada.")

Função de extração principal carregada.


## 4. Caminhos dos Documentos

In [8]:
BASE = "/home/anab/Documentos/Documentos importantes/Desafio Técnico - Estágio em P&D/Documentos (1)/"

PATH_CNH    = BASE + "Documento 1.jpeg"
PATH_FATURA = BASE + "Documento 2.jpg"
PATH_PAPER  = BASE + "Documento 3.pdf"

print(f"CNH    existe: {Path(PATH_CNH).exists()}")
print(f"Fatura existe: {Path(PATH_FATURA).exists()}")
print(f"Paper  existe: {Path(PATH_PAPER).exists()}")

CNH    existe: True
Fatura existe: True
Paper  existe: True


## 5. Caso 1 — Documento Estruturado (CNH)

**Objetivo:** Extrair campos predeterminados de uma CNH em uma única inferência com saída JSON validada.

**Campos extraídos:** nome, CPF, data de nascimento, data de emissão, filiação pai, filiação mãe, número de registro, categoria, validade, local de emissão.

> `thinking_budget=0` — extração simples de campos fixos; raciocínio incremental desnecessário.

In [9]:
schema_cnh = {
    "type": "object",
    "properties": {
        "nome":            {"type": "string",  "description": "Nome completo do titular"},
        "cpf":             {"type": "string",  "description": "CPF no formato XXX.XXX.XXX-XX"},
        "data_nascimento": {"type": "string",  "description": "Data de nascimento DD/MM/AAAA"},
        "data_emissao":    {"type": "string",  "description": "Data de emissão da CNH"},
        "filiacao_pai":    {"type": "string",  "description": "Nome do pai"},
        "filiacao_mae":    {"type": "string",  "description": "Nome da mãe"},
        "numero_registro": {"type": "string",  "description": "Número de registro"},
        "categoria":       {"type": "string",  "description": "Categoria da habilitação"},
        "validade":        {"type": "string",  "description": "Data de validade"},
        "local_emissao":   {"type": "string",  "description": "Local de emissão"}
    },
    "required": ["nome", "cpf", "data_nascimento", "data_emissao", "filiacao_pai", "filiacao_mae"]
}

dados_cnh, mime_cnh = carregar_imagem(PATH_CNH)

conteudo_cnh = [
    types.Part.from_bytes(data=dados_cnh, mime_type=mime_cnh),
    "Extraia todos os campos visíveis desta CNH brasileira. Para campos ilegíveis ou ausentes, retorne null."
]

resultado_cnh, metricas_cnh = extrair_com_schema(
    conteudo_cnh, schema_cnh, "CASO 1 — CNH", thinking_budget=0
)

print("\nResultado:")
print(json.dumps(resultado_cnh, ensure_ascii=False, indent=2))


  CASO 1 — CNH
Tentativa 1/3 (thinking_budget=0)...
  Latência:      2.87s
  Tokens input:  284
  Tokens output: 136
  Custo est.:    $0.000124 USD
  Inferências:   1 (pipeline unificado)

Resultado:
{
  "nome": "LINCE DA SILVA",
  "cpf": "891.340.611-75",
  "data_nascimento": "02/05/2017",
  "data_emissao": "22/10/2013",
  "filiacao_pai": "Pai José da Silva",
  "filiacao_mae": "Mãe Maria da Silva",
  "numero_registro": "00123456789",
  "categoria": "B",
  "validade": "02/05/2018",
  "local_emissao": "BRASÍLIA - DISTRITO FEDERAL, DF"
}


## 6. Caso 2 — Documento Extenso (Claude 3 Model Family)

**Objetivo:** Extrair o conteúdo de um paper técnico de 42 páginas preservando a estrutura das tabelas e interpretando os gráficos.

> **Nota metodológica:** Por limitação de tamanho da requisição, foram enviadas as primeiras 10 páginas (de 42 no total). Em produção, a janela de contexto de 1M tokens do Gemini 2.5 Flash suporta o documento completo sem fragmentação.

> `thinking_budget=1024` — documento técnico denso; raciocínio incremental ativo para maior acurácia na interpretação de gráficos.

In [10]:
schema_paper = {
    "type": "object",
    "properties": {
        "titulo":   {"type": "string"},
        "abstract": {"type": "string"},
        "modelos": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "nome":      {"type": "string"},
                    "descricao": {"type": "string"}
                }
            }
        },
        "principais_benchmarks": {
            "type": "array",
            "description": "Até 5 benchmarks mais relevantes com scores resumidos",
            "items": {
                "type": "object",
                "properties": {
                    "nome":   {"type": "string"},
                    "resumo": {"type": "string"}
                }
            }
        },
        "graficos": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "figura":    {"type": "string"},
                    "conclusao": {"type": "string"}
                }
            }
        },
        "secoes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "titulo": {"type": "string"},
                    "resumo": {"type": "string"}
                }
            }
        }
    },
    "required": ["titulo", "abstract", "modelos", "principais_benchmarks", "graficos", "secoes"]
}

dados_paper = carregar_pdf_parcial(PATH_PAPER, max_paginas=10)

conteudo_paper = [
    types.Part.from_bytes(data=dados_paper, mime_type="application/pdf"),
    """Analise este paper técnico e extraia o conteúdo estruturado.
Instruções:
1. Preserve todos os dados numéricos das tabelas exatamente como aparecem
2. Para cada gráfico, descreva a principal conclusão
3. Extraia os resultados comparativos entre os modelos
4. Mantenha a hierarquia de seções"""
]

resultado_paper, metricas_paper = extrair_com_schema(
    conteudo_paper, schema_paper, "CASO 2 — CLAUDE 3 PAPER",
    thinking_budget=1024, max_output_tokens=16000
)

print("\nResumo do resultado:")
print(json.dumps({
    "titulo":             resultado_paper.get("titulo"),
    "num_modelos":        len(resultado_paper.get("modelos", [])),
    "num_benchmarks":     len(resultado_paper.get("principais_benchmarks", [])),
    "num_graficos":       len(resultado_paper.get("graficos", [])),
    "num_secoes":         len(resultado_paper.get("secoes", []))
}, ensure_ascii=False, indent=2))

print("\nModelos extraídos:")
print(json.dumps(resultado_paper.get("modelos", []), ensure_ascii=False, indent=2))

Páginas enviadas: 10 de 42

  CASO 2 — CLAUDE 3 PAPER
Tentativa 1/3 (thinking_budget=1024)...
  Latência:      21.61s
  Tokens input:  2651
  Tokens output: 3146
  Custo est.:    $0.002285 USD
  Inferências:   1 (pipeline unificado)

Resumo do resultado:
{
  "titulo": "The Claude 3 Model Family: Opus, Sonnet, Haiku",
  "num_modelos": 3,
  "num_benchmarks": 5,
  "num_graficos": 1,
  "num_secoes": 14
}

Modelos extraídos:
[
  {
    "nome": "Claude 3 Opus",
    "descricao": "O modelo mais capaz e inteligente da família Claude 3. Demonstra resultados de ponta em raciocínio, matemática e codificação, superando outros modelos em avaliações como GPQA, MMLU e MMMU."
  },
  {
    "nome": "Claude 3 Sonnet",
    "descricao": "Oferece uma combinação de habilidades e velocidade. Apresenta proficiência aumentada na criação de conteúdo matizado, análise, previsão, sumarização precisa e tratamento de consultas científicas."
  },
  {
    "nome": "Claude 3 Haiku",
    "descricao": "O modelo mais rápido 

## 7. Caso 3 — Documento com Layout Complexo (Fatura de Energia)

**Objetivo:** Extrair o conteúdo da fatura CELPE preservando a organização das múltiplas seções tabulares, tributos e histórico de consumo.

> `thinking_budget=0` — layout complexo mas campos bem definidos; extração direta.

In [11]:
schema_fatura = {
    "type": "object",
    "properties": {
        "dados_cliente": {
            "type": "object",
            "properties": {
                "nome":          {"type": "string"},
                "cpf":           {"type": "string"},
                "endereco":      {"type": "string"},
                "classificacao": {"type": "string"}
            }
        },
        "dados_fatura": {
            "type": "object",
            "properties": {
                "conta_contrato":  {"type": "string"},
                "mes_ano":         {"type": "string"},
                "data_vencimento": {"type": "string"},
                "total_a_pagar":   {"type": "number"}
            }
        },
        "consumo": {
            "type": "object",
            "properties": {
                "leitura_anterior": {"type": "number"},
                "leitura_atual":    {"type": "number"},
                "consumo_kwh":      {"type": "number"},
                "numero_dias":      {"type": "integer"}
            }
        },
        "tributos": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "tributo":      {"type": "string"},
                    "aliquota_pct": {"type": "number"},
                    "valor":        {"type": "number"}
                }
            }
        },
        "composicao_valor": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "item":       {"type": "string"},
                    "valor_rs":   {"type": "number"},
                    "percentual": {"type": "number"}
                }
            }
        },
        "historico_consumo": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "mes_ano":     {"type": "string"},
                    "consumo_kwh": {"type": "number"}
                }
            }
        }
    },
    "required": ["dados_cliente", "dados_fatura", "consumo", "tributos"]
}

dados_fatura, mime_fatura = carregar_imagem(PATH_FATURA)

conteudo_fatura = [
    types.Part.from_bytes(data=dados_fatura, mime_type=mime_fatura),
    """Extraia todas as informações desta fatura de energia elétrica.
Instruções:
1. Extraia dados do cliente, dados da fatura e consumo
2. Extraia os tributos (ICMS, PIS, COFINS) com alíquotas e valores
3. Extraia a composição do valor com percentuais
4. Extraia o histórico de consumo mensal do gráfico de barras
5. Converta vírgula para ponto nos valores numéricos"""
]

resultado_fatura, metricas_fatura = extrair_com_schema(
    conteudo_fatura, schema_fatura, "CASO 3 — FATURA DE ENERGIA", thinking_budget=0
)

print("\nResultado:")
print(json.dumps(resultado_fatura, ensure_ascii=False, indent=2))


  CASO 3 — FATURA DE ENERGIA
Tentativa 1/3 (thinking_budget=0)...
  Latência:      4.11s
  Tokens input:  357
  Tokens output: 686
  Custo est.:    $0.000465 USD
  Inferências:   1 (pipeline unificado)

Resultado:
{
  "dados_cliente": {
    "nome": "MARIA JOSÉ DA SILVA",
    "cpf": "123.456.789-10",
    "endereco": "RUA JOÃO FERNANDES VIEIRA 175 BOA VISTA",
    "classificacao": "BI RESIDENCIAL RESIDENCIAL Monofásico"
  },
  "dados_fatura": {
    "conta_contrato": "1234567890",
    "mes_ano": "06/2014",
    "data_vencimento": "10/07/2014",
    "total_a_pagar": 63.72
  },
  "consumo": {
    "leitura_anterior": 24630.0,
    "leitura_atual": 24790.0,
    "consumo_kwh": 160.0,
    "numero_dias": 31
  },
  "tributos": [
    {
      "tributo": "ICMS",
      "aliquota_pct": 17.0,
      "valor": 10.83
    },
    {
      "tributo": "PIS",
      "aliquota_pct": 0.89,
      "valor": 0.83
    },
    {
      "tributo": "COFINS",
      "aliquota_pct": 4.54,
      "valor": 2.89
    }
  ],
  "composic

## 8. Análise Comparativa dos Resultados

Consolidação das métricas dos 3 casos para inclusão na Seção 4.3 do Dossiê de Pesquisa.

In [12]:
print("=" * 70)
print("RESUMO DE MÉTRICAS — POC EXTRAÇÃO ESTRUTURADA COM GEMINI 2.5 FLASH")
print("=" * 70)
print(f"{'Caso':<30} {'Latência':>9} {'T.Input':>8} {'T.Output':>9} {'Custo USD':>11}")
print("-" * 70)

casos = [
    ("Caso 1 — CNH",               metricas_cnh),
    ("Caso 2 — Claude 3 Paper",    metricas_paper),
    ("Caso 3 — Fatura de Energia", metricas_fatura),
]

total_custo    = 0.0
total_latencia = 0.0

for nome, m in casos:
    print(
        f"{nome:<30} "
        f"{m['latencia_s']:>8.2f}s "
        f"{m['tokens_input']:>8} "
        f"{m['tokens_output']:>9} "
        f"${m['custo_usd']:>10.6f}"
    )
    total_custo    += m['custo_usd']
    total_latencia += m['latencia_s']

print("-" * 70)
print(f"{'TOTAL':<30} {total_latencia:>8.2f}s {'':>8} {'':>9} ${total_custo:>10.6f}")
print("=" * 70)
print(f"\nTodas as extrações: 1 inferência por documento.")
print(f"Pipeline atual estimado: 2 inferências × latência e custo.")
print(f"Redução estimada vs. pipeline atual: ~50–60% em custo e latência.")

RESUMO DE MÉTRICAS — POC EXTRAÇÃO ESTRUTURADA COM GEMINI 2.5 FLASH
Caso                            Latência  T.Input  T.Output   Custo USD
----------------------------------------------------------------------
Caso 1 — CNH                       2.87s      284       136 $  0.000124
Caso 2 — Claude 3 Paper           21.61s     2651      3146 $  0.002285
Caso 3 — Fatura de Energia         4.11s      357       686 $  0.000465
----------------------------------------------------------------------
TOTAL                             28.59s                    $  0.002874

Todas as extrações: 1 inferência por documento.
Pipeline atual estimado: 2 inferências × latência e custo.
Redução estimada vs. pipeline atual: ~50–60% em custo e latência.


## 9. Exportação dos Resultados

In [13]:
output = {
    "modelo": MODEL,
    "caso_1_cnh": {
        "resultado": resultado_cnh,
        "metricas":  metricas_cnh
    },
    "caso_2_paper": {
        "resultado": resultado_paper,
        "metricas":  metricas_paper
    },
    "caso_3_fatura": {
        "resultado": resultado_fatura,
        "metricas":  metricas_fatura
    }
}

with open("resultados_poc.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("Resultados exportados para resultados_poc.json")

Resultados exportados para resultados_poc.json
